In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
dtype_mapping = {
    "Identificación": "string",
    "ID incidente padre": "string",
    "Alimentador normal": "string",
    "Alimentador actual": "string",
    "Ubicación de la incidencia": "string",
    "Dirección del dispositivo": "string",
    "Usuario asignado": "string",
    "Cuadrillas": "string",
    "Dispositivo": "string",
    "Tipo de dispositivo": "string",
    "Problema": "string",
    "Motivo de cancelación": "string",
    "Causa": "string",
    "Subcausa": "string",
    "Comentario": "string",
    "Tiempo creado": "datetime64[ns]",
    "Fecha de interrupción": "datetime64[ns]",
    "ETR": "datetime64[ns]",
    "ATR": "datetime64[ns]",
    "ETA": "datetime64[ns]",
    "ATA": "datetime64[ns]",
    "Fecha de Cierre": "datetime64[ns]",
    "Afectados": "Int64",
    "Críticos afectados": "Int64",
    "Llamadas": "Int64",
    "Tensión [kV]": "Float64",
    "Dispositivo aguas arriba": "string",
    "Instrucción": "string",
    "Tipo": "string",
    "Subtipo": "string",
    "Confirmado": "string",
    "Prioridad": "Int64",
    "Duración de incidencia [min]": "Float64",
    "Horas de interrupción del cliente": "Float64",
    "ENSI [kWh/intervalo]": "Float64",
    "Subestación": "string",
    "Región": "string",
    "Subregión": "string",
    "Marcación": "string",
    "Trámite FM": "string",
    "Relé/Fusible": "string",
    "Tipo de protección": "string",
    "Está enlazado": "string",
    "Tiene enlaces": "string",
    "CODOSI": "string",
    "Observaciones Finales": "string",
    "Condición de Falla": "string",
    "Loc. Genérica": "string",
    "Loc. Específica": "string",
    "Distrito": "string",
    "Zona": "string",
    "Objeto de Alimentador": "string",
    "Nodo de Referencia": "string",
    "LDS Priority": "string",
    "Llamadas Reiterativas": "Int64",
    "Tiempo transcurrido [min]": "Float64",
    "Tiempo excedido [min]": "Float64",
    "Plazo": "string",
    "horas": "Float64",
    "Horas cliente": "Float64"
}

In [ ]:
# Verificamos que los nombres del mapeo del tipo de dato de las columnas coinciden con el Excel
excel_cols = pd.read_excel("D:/Proyectos/consumo-electrico/data/raw/Incidencias BT al 02_2026.xlsx", sheet_name="Tabla", nrows=0).columns.tolist()
mapeo_cols = list(dtype_mapping.keys())

no_coinciden = [c for c in mapeo_cols if c not in excel_cols]
print(f"Columnas del mapeo que NO están en el Excel: {no_coinciden}")

In [ ]:
#Literalmente reemplazamos 'nan' por None para evitar problemas al cargar el DataFrame
df_raw=pd.read_excel("D:/Proyectos/consumo-electrico/data/raw/Incidencias BT al 02_2026.xlsx", sheet_name="Tabla")
df_raw=df_raw.replace('nan', None)

In [ ]:
#Chequeamos que todo, efectivamente, esté en string
# Ver cuántas filas y columnas cargamos
print(f"Filas: {df_raw.shape[0]}")
print(f"Columnas: {df_raw.shape[1]}")
print(f"\nTipos actuales (todos deben ser object):")
print(df_raw.dtypes.value_counts())

In [ ]:


# Aplicar conversiones del mapeo
for col, dtype in dtype_mapping.items():
    if col in df_raw.columns:
        try:
            if dtype == 'datetime64[ns]':
                df_raw[col] = pd.to_datetime(df_raw[col], errors='coerce')
            elif dtype in ('int64', 'int32'):
                df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').astype(dtype)
            elif dtype in ('float64', 'float32'):
                df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').astype(dtype)
            else:
                df_raw[col] = df_raw[col].astype(dtype)
            print(f"✓ {col}: convertido a {dtype}")
        except Exception as e:
            print(f"✗ {col}: error en conversión - {e}")

print(f"\nTipos finales (Opción 3):")
print(df_raw.dtypes)

In [ ]:
#Revisamos que todo está correcto y no hay columnas que quedaron sin convertir
print("Tipos finales:")
print(df_raw.dtypes.value_counts())
print(f"\nNulos por columna (top 10):")
print(df_raw.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
#Comprobamos que la columna "Fecha de Cierre" se convirtió correctamente a datetime y contamos los nulos
#Todo con la finalidad de verificar que la lógica se pueda sostener a lo largo de todo el proceso
print(df_raw["Fecha de Cierre"].isnull().sum())

In [ ]:
#Bien, hay problemas con la forma de calcular la duración de la incidencia, ya que hay valores negativos. 
# Veamos si esta columna nos puede salvar
c=0
for i in df_raw["Duración de incidencia [min]"]:
    if 0 > i:
        c+=1
print(f"Cantidad de filas con duración negativa: {c}")

In [ ]:
# Usar duración existente cuando es válida, híbrida cuando es negativa o nula
df_raw['duracion_hibrida'] = (
    df_raw['ATR'].fillna(df_raw['Fecha de Cierre']) - df_raw['Fecha de interrupción']
).dt.total_seconds() / 60

df_raw['duracion_final'] = df_raw['Duración de incidencia [min]'].where(
    df_raw['Duración de incidencia [min]'] > 0,
    df_raw['duracion_hibrida']
)

# Verificar
print(f"Negativos en duracion_final: {(df_raw['duracion_final'] < 0).sum()}")
print(f"Nulos en duracion_final: {df_raw['duracion_final'].isnull().sum()}")
print(df_raw['duracion_final'].describe())

In [ ]:
#Bueno, sí hay problemas con la duración ya que hay negativos, csmre.
# Ver los registros con duración negativa
negativos = df_raw[df_raw['duracion_final'] < 0][
    ['Identificación', 'Fecha de interrupción', 'ATR', 
     'Fecha de Cierre', 'Duración de incidencia [min]', 'duracion_final']
]
print(f"Registros con duración negativa: {len(negativos)}")
print(negativos.head(10))

In [ ]:
print(df_raw[df_raw['duracion_final'] < 0]['Afectados'].value_counts())
# Usar duración oficial y simplemente eliminar los negativos
df_clean = df_raw[df_raw['Duración de incidencia [min]'] > 0].copy()
print(f"Registros eliminados: {227852 - len(df_clean)}")

## Decisión: eliminación de registros con duración negativa
- 68 registros eliminados de 227,852 (0.03%)
- Causa: errores de registro de fechas en el sistema origen
- Impacto: estadísticamente irrelevante
- Dataset limpio: 227,784 registros

In [ ]:
print(f"Dataset limpio: {len(df_clean)} registros")
print(f"Eliminados: {227852 - len(df_clean)}")
print(f"\nNegativos restantes: {(df_clean['Duración de incidencia [min]'] < 0).sum()}")
print(f"Nulos en duración: {df_clean['Duración de incidencia [min]'].isnull().sum()}")

Revisamos que la columna de duracion final tenga sentido, temo por los outliers

In [ ]:

df_clean["duracion_final"].describe()

In [ ]:
print(df_clean['Duración de incidencia [min]'].describe())
print(f"\nRegistros con duración > 10,000 min: {(df_clean['Duración de incidencia [min]'] > 10000).sum()}")
print(f"Registros con duración > 1,440 min (24h): {(df_clean['Duración de incidencia [min]'] > 1440).sum()}")
print(f"Registros con duración > 480 min (8h): {(df_clean['Duración de incidencia [min]'] > 480).sum()}")

In [ ]:
#Bien, veamos qué particularidades tienen los registros con duración mayor a 10,000 minutos
outliers = df_clean[df_clean['Duración de incidencia [min]'] > 10000][
    ['Identificación', 'Fecha de interrupción', 'Duración de incidencia [min]', 
     'Zona', 'Distrito', 'Causa']
].sort_values('Duración de incidencia [min]', ascending=False)

print(outliers)

In [ ]:
df_clean = df_clean[df_clean['Duración de incidencia [min]'] <= 10000].copy()

print(f"Registros después de eliminar outliers extremos: {len(df_clean)}")
print(f"Eliminados: {227784 - len(df_clean)}")

In [ ]:
print(df_clean['Duración de incidencia [min]'].describe())
print("------")
print(df_clean["duracion_final"].describe())

Una vez comprobado la limpieza, pasamos el df a parquet para aumentar la eficiencia de la manipulación de datos

In [ ]:
import os

os.makedirs("D:/Proyectos/consumo-electrico/data/processed", exist_ok=True)

df_clean.to_parquet(
    "D:/Proyectos/consumo-electrico/data/processed/incidencias_clean.parquet",
    index=False
)

size = os.path.getsize("D:/Proyectos/consumo-electrico/data/processed/incidencias_clean.parquet")
print(f"Guardado exitosamente.")
print(f"Registros: {len(df_clean):,}")
print(f"Tamaño: {size / 1024 / 1024:.1f} MB")